In [1]:
import os
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

try:
    from nltk.tokenize import TweetTokenizer

    _HAS_NLTK = True
except Exception:
    _HAS_NLTK = False

# Fix: remove TensorFlow usage entirely to avoid Kaggle environment TF/protobuf/Keras3 crashes:
# - Cell 0 previously failed on import with protobuf MessageFactory error.
# - Later cells failed due to `tf.compat.v1.layers.dense` not available with Keras 3.
# We keep identical core logic: TF-IDF features -> 2 hidden layers + dropout -> 5-class softmax,
# trained with Adam and cross-entropy for a fixed number of epochs.

np.random.seed(42)



In [2]:
CANDIDATE_BASES = [
    "/kaggle/input/movie-review-sentiment-analysis-kernels-only",
    "/kaggle/data/movie-review-sentiment-analysis-kernels-only",
    "/kaggle/input",
    "/kaggle/data",
]


def find_file(filename):
    for base in CANDIDATE_BASES:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"Could not find {filename} under any of: {CANDIDATE_BASES}"
    )


train_path = find_file("train.tsv")
test_path = find_file("test.tsv")

train_data = pd.read_csv(train_path, sep="\t")
test_data = pd.read_csv(test_path, sep="\t")

print(train_data.shape, test_data.shape)
print(train_data.columns.tolist())



(109242, 4) (46818, 3)
['PhraseId', 'SentenceId', 'Phrase', 'Sentiment']


In [3]:
assert {"PhraseId", "Phrase", "Sentiment"}.issubset(set(train_data.columns))
assert {"PhraseId", "Phrase"}.issubset(set(test_data.columns))
assert train_data["Sentiment"].between(0, 4).all()



In [4]:
train_data.head()



,PhraseId,SentenceId,Phrase,Sentiment
0,24144,1097,through every frame,2
1,85111,4402,onscreen presence,2
2,75999,3900,into dysfunction,2
3,17377,754,a probe,2
4,9664,403,half hour,2


In [5]:
train_data["SentenceId"].unique().size



8503

In [6]:
corpus = pd.concat(
    [train_data["Phrase"], test_data["Phrase"]], axis=0, ignore_index=True
).values



In [7]:
assert corpus.shape[0] == train_data["Phrase"].shape[0] + test_data["Phrase"].shape[0]




In [8]:
def tokenize(x):
    if x is None:
        return []
    x = str(x)
    if len(x) == 1:
        return [x]
    if _HAS_NLTK:
        return TweetTokenizer().tokenize(x)
    return x.split()


tfVectorizer = TfidfVectorizer(tokenizer=tokenize)
tf_idf_total = tfVectorizer.fit_transform(corpus)



/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [9]:
tf_idf_total = tf_idf_total.todense()
tf_idf_total.shape



(156060, 16395)

In [10]:
tf_idf_dense = tf_idf_total[: train_data["Phrase"].shape[0]]
tf_idf_dense.shape



(109242, 16395)

In [11]:
tf_idf_dense.shape



(109242, 16395)

In [12]:
tfVectorizer.get_params()



{'analyzer': 'word',
 'binary': False,
 'decode_error': 'strict',
 'dtype': numpy.float64,
 'encoding': 'utf-8',
 'input': 'content',
 'lowercase': True,
 'max_df': 1.0,
 'max_features': None,
 'min_df': 1,
 'ngram_range': (1, 1),
 'norm': 'l2',
 'preprocessor': None,
 'smooth_idf': True,
 'stop_words': None,
 'strip_accents': None,
 'sublinear_tf': False,
 'token_pattern': '(?u)\\b\\w\\w+\\b',
 'tokenizer': <function __main__.tokenize(x)>,
 'use_idf': True,
 'vocabulary': None}

In [13]:
train_vocab = getattr(tfVectorizer, "vocabulary_", None)
type(train_vocab), (len(train_vocab) if train_vocab is not None else None)



(dict, 16395)

In [14]:
if hasattr(tfVectorizer, "get_feature_names_out"):
    feature_names = tfVectorizer.get_feature_names_out()
else:
    feature_names = tfVectorizer.get_feature_names()
feature_names[:10]




array([' ', '!', '#', '$', '&', "'", '*', '+', ',', '-'], dtype=object)

In [15]:
def top_tfidf_words(row, features, top_n=20):
    topn_ids = np.argsort(row)[::-1][:top_n]
    top_feats = [(features[i], row[i]) for i in topn_ids]
    df = pd.DataFrame(top_feats)
    df.columns = ["feature", "tfidf"]
    return df


def top_words_in_doc(Xtr, features, row_id, top_n=20):
    row = np.squeeze(np.asarray(Xtr[row_id]))
    return top_tfidf_words(row, features, top_n)


def top_mean_words(Xtr, features, grp_ids=None, min_tfidf=0.1, top_n=10):
    if grp_ids:
        D = np.asarray(Xtr[grp_ids])
    else:
        D = np.asarray(Xtr)

    D[D < min_tfidf] = 0
    tfidf_means = np.mean(D, axis=0)
    return top_tfidf_words(tfidf_means, features, top_n)


def top_words_by_class(Xtr, y, features, min_tfidf=0.1, top_n=20):
    dfs = []
    labels_ = np.unique(y)
    for label in labels_:
        ids = np.where(y == label)
        feats_df = top_mean_words(Xtr, features, ids, min_tfidf=min_tfidf, top_n=top_n)
        feats_df.label = label
        dfs.append(feats_df)
    return dfs


def plot_tfidf_classWords_h(dfs, num_class=9):
    fig = plt.figure(figsize=(12, 100), facecolor="w")
    x = np.arange(len(dfs[0]))
    for i, df in enumerate(dfs):
        ax = fig.add_subplot(num_class, 1, i + 1)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_frame_on(False)
        ax.get_xaxis().tick_bottom()
        ax.get_yaxis().tick_left()
        ax.set_xlabel("Mean Tf-Idf Score", labelpad=16, fontsize=16)
        ax.set_ylabel("Word", labelpad=16, fontsize=16)
        ax.set_title("Class = " + str(df.label), fontsize=25)
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 2))
        ax.barh(x, df.tfidf, align="center")
        ax.set_yticks(x)
        ax.set_ylim([-1, x[-1] + 1])
        ax.set_yticklabels(df.feature)

        for tick in ax.yaxis.get_major_ticks():
            tick.label.set_fontsize(20)
        plt.subplots_adjust(bottom=0.09, right=0.97, left=0.15, top=0.95, wspace=0.52)
    plt.show()




In [16]:
def get_batch(index, batch_size=2000):
    index = shuffle(index, random_state=42)
    batch_index = []
    for sample in index:
        batch_index.append(sample)
        if len(batch_index) == batch_size:
            yield batch_index
            batch_index = []
    if len(batch_index) > 0:
        yield batch_index




In [17]:
tf_idf_dense2 = tf_idf_total[train_data["Phrase"].shape[0] :]
assert len(tf_idf_dense) + len(tf_idf_dense2) == len(tf_idf_total)
tf_idf_dense2.shape



(46818, 16395)

In [18]:
test_data.head()



,PhraseId,SentenceId,Phrase
0,21924,980,escape movie
1,18207,795,well-executed
2,81803,4221,flick formula
3,114983,6124,is too savvy a filmmaker to let this morph int...
4,135928,7343,"There are laughs aplenty ,"


In [19]:
tf_idf_dense2 = tf_idf_total[train_data["Phrase"].shape[0] :]
tf_idf_dense2.shape



(46818, 16395)

In [20]:
assert len(tf_idf_dense) + len(tf_idf_dense2) == len(tf_idf_total)



In [21]:
pass



In [22]:
pass



In [23]:
pass



In [24]:
# Fix: replace TF graph with a NumPy MLP matching the same architecture:
# input -> Dense(50, relu) -> Dropout(keep=0.7) -> Dense(30, relu) -> Dropout(keep=0.8) -> Dense(5) -> softmax
# Loss: softmax cross-entropy. Optimizer: Adam. Training=True dropout behavior.
# This preserves the core modeling logic and keeps runtime stable.


def relu(x):
    return np.maximum(x, 0.0)


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / (np.sum(ez, axis=1, keepdims=True) + 1e-12)


def one_hot(y, num_classes=5):
    out = np.zeros((y.shape[0], num_classes), dtype=np.float32)
    out[np.arange(y.shape[0]), y] = 1.0
    return out


def dropout_forward(x, keep_prob, rng):
    if keep_prob >= 1.0:
        return x, None
    mask = (rng.rand(*x.shape) < keep_prob).astype(np.float32) / keep_prob
    return x * mask, mask


class AdamState:
    def __init__(self, shape):
        self.m = np.zeros(shape, dtype=np.float32)
        self.v = np.zeros(shape, dtype=np.float32)


def adam_update(param, grad, state: AdamState, lr, beta1, beta2, eps, t):
    state.m = beta1 * state.m + (1.0 - beta1) * grad
    state.v = beta2 * state.v + (1.0 - beta2) * (grad * grad)
    m_hat = state.m / (1.0 - beta1**t)
    v_hat = state.v / (1.0 - beta2**t)
    param -= lr * m_hat / (np.sqrt(v_hat) + eps)




In [25]:
tf_idf_dense = np.asarray(tf_idf_dense, dtype=np.float32)
tf_idf_dense2 = np.asarray(tf_idf_dense2, dtype=np.float32)
y_all = train_data["Sentiment"].values.astype(np.int32)

print(tf_idf_dense.shape, tf_idf_dense2.shape, y_all.shape)



(109242, 16395) (46818, 16395) (109242,)


In [26]:
x_train, x_vad, y_train, y_vad = train_test_split(
    tf_idf_dense, y_all, test_size=0.2, random_state=42, stratify=y_all
)



In [27]:
print(x_train.shape)
print(x_vad.shape)



(87393, 16395)
(21849, 16395)


In [28]:
(
    i,
    j,
    max_epoch,
    patience,
    loss_max,
) = (
    0,
    0,
    50,
    5,
    np.inf,
)
n = 1
optimal_i = i



In [29]:
# Keep the same hard-coded epoch count used by the original notebook.
optimal_i = 38

# Model hyperparams mirroring the TF setup
n_sentance, n_words = tf_idf_dense.shape
h1, h2, n_classes = 50, 30, 5
keep1, keep2 = 0.7, 0.8

# Adam params (TensorFlow defaults)
lr = 0.001
beta1, beta2, eps = 0.9, 0.999, 1e-7

rng = np.random.RandomState(42)

# He/Glorot-like init (Dense default in Keras differs; keep reasonable stable init)
W1 = (rng.randn(n_words, h1).astype(np.float32)) * np.sqrt(2.0 / max(1, n_words))
b1 = np.zeros((h1,), dtype=np.float32)
W2 = (rng.randn(h1, h2).astype(np.float32)) * np.sqrt(2.0 / max(1, h1))
b2 = np.zeros((h2,), dtype=np.float32)
W3 = (rng.randn(h2, n_classes).astype(np.float32)) * np.sqrt(1.0 / max(1, h2))
b3 = np.zeros((n_classes,), dtype=np.float32)

# Adam states
W1_s, b1_s = AdamState(W1.shape), AdamState(b1.shape)
W2_s, b2_s = AdamState(W2.shape), AdamState(b2.shape)
W3_s, b3_s = AdamState(W3.shape), AdamState(b3.shape)

t = 0
index = np.arange(len(tf_idf_dense))

for epoch in range(optimal_i):
    # Shuffle indices similarly to original get_batch usage
    for batch_index in list(get_batch(index, batch_size=2000)):
        X = tf_idf_dense[batch_index]
        y = y_all[batch_index]
        Y = one_hot(y, num_classes=n_classes)

        # Forward
        z1 = X @ W1 + b1
        a1 = relu(z1)
        a1d, m1 = dropout_forward(a1, keep1, rng)

        z2 = a1d @ W2 + b2
        a2 = relu(z2)
        a2d, m2 = dropout_forward(a2, keep2, rng)

        logits = a2d @ W3 + b3
        probs = softmax(logits)

        # Loss
        loss_value = -np.mean(np.sum(Y * np.log(probs + 1e-12), axis=1))

        # Backward (softmax + cross-entropy)
        dlogits = (probs - Y) / X.shape[0]  # mean over batch

        dW3 = a2d.T @ dlogits
        db3 = np.sum(dlogits, axis=0)

        da2d = dlogits @ W3.T
        if m2 is not None:
            da2 = da2d * m2
        else:
            da2 = da2d
        dz2 = da2 * (z2 > 0).astype(np.float32)

        dW2 = a1d.T @ dz2
        db2 = np.sum(dz2, axis=0)

        da1d = dz2 @ W2.T
        if m1 is not None:
            da1 = da1d * m1
        else:
            da1 = da1d
        dz1 = da1 * (z1 > 0).astype(np.float32)

        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0)

        # Update
        t += 1
        adam_update(W3, dW3.astype(np.float32), W3_s, lr, beta1, beta2, eps, t)
        adam_update(b3, db3.astype(np.float32), b3_s, lr, beta1, beta2, eps, t)
        adam_update(W2, dW2.astype(np.float32), W2_s, lr, beta1, beta2, eps, t)
        adam_update(b2, db2.astype(np.float32), b2_s, lr, beta1, beta2, eps, t)
        adam_update(W1, dW1.astype(np.float32), W1_s, lr, beta1, beta2, eps, t)
        adam_update(b1, db1.astype(np.float32), b1_s, lr, beta1, beta2, eps, t)

    # Report training accuracy on last batch of the epoch (similar to original print)
    pred_last = np.argmax(probs, axis=1).astype(np.int32)
    accuracy = float(np.mean(pred_last == y))
    print(f"accuracy for epoch {epoch} is {accuracy}, loss is {loss_value}")

# Inference on test set (no dropout)
Xtest = tf_idf_dense2
z1 = Xtest @ W1 + b1
a1 = relu(z1)
z2 = a1 @ W2 + b2
a2 = relu(z2)
logits = a2 @ W3 + b3
pred_value = np.argmax(logits, axis=1).astype(np.int32)

sentiment = test_data[["PhraseId"]].copy()
sentiment["Sentiment"] = pred_value
sentiment.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", pd.read_csv("submission.csv").shape)
print(pd.read_csv("submission.csv").head())

accuracy for epoch 0 is 0.5193236714975845, loss is 1.3436861038208008
accuracy for epoch 1 is 0.5619967793880838, loss is 1.1632163524627686
accuracy for epoch 2 is 0.607085346215781, loss is 0.9931839108467102
accuracy for epoch 3 is 0.6561996779388084, loss is 0.8685349225997925
accuracy for epoch 4 is 0.6892109500805152, loss is 0.7818590998649597
accuracy for epoch 5 is 0.7077294685990339, loss is 0.732804536819458
accuracy for epoch 6 is 0.7230273752012882, loss is 0.6824703216552734
accuracy for epoch 7 is 0.7391304347826086, loss is 0.6482226848602295
accuracy for epoch 8 is 0.7479871175523349, loss is 0.6092258095741272
accuracy for epoch 9 is 0.7544283413848631, loss is 0.5890878438949585
accuracy for epoch 10 is 0.7681159420289855, loss is 0.5680137872695923
accuracy for epoch 11 is 0.7745571658615137, loss is 0.5471447706222534
accuracy for epoch 12 is 0.7962962962962963, loss is 0.5166538953781128
accuracy for epoch 13 is 0.8027375201288245, loss is 0.5074472427368164
accu